<a href="https://colab.research.google.com/github/GuruShrihari/IntroToML/blob/main/MAGIC_Gamma_Telescope.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler

In [ ]:
cols = ["fLength","fWidth","fSize","fConc","fConc1","fAsym","fM3Long","fM3Trans","fAlpha","fDist","class"]
df = pd.read_csv("magic04.data",names = cols)

df['class'].unique()

In [ ]:
#Since the computer understands numbers better than letters we convert the class into 0 or 1

df["class"] = (df["class"] == "g").astype(int) #Converts the column into integer and makes all g as 1 and all h as 0 since there are only 2 types of letters in the class column


In [ ]:
df.head()


In [ ]:
for labels in cols:
  plt.hist(df[df["class"] == 1][labels],color="blue", label="gamma",alpha= 0.7, density=True)
  plt.hist(df[df["class"] == 0][labels],color="red", label="hydron",alpha= 0.7, density=True)
  plt.title(labels)
  plt.xlabel(labels)
  plt.ylabel("Probability")
  plt.legend()
  plt.show()

# **Train, validation and Test datasets**

In [ ]:
train,valid,test = np.split(df.sample(frac=1),[int(0.6 * len(df)),int(0.8 * len(df))])


#Now As you can see some columns have data in the hundereds while others are in the 0.001s this extremely affects data predictions and this can be fixed using StandardScaler from sklearn

def scale_dataset(dataframe, oversample = False):
  x = dataframe[dataframe.columns[:-1]].values
  y = dataframe[dataframe.columns[-1]].values

  scaler = StandardScaler()
  x = scaler.fit_transform(x)
  if oversample:
    ros = RandomOverSampler()
    x,y = ros.fit_resample(x,y) #This takes the class which has lesser data and then samples till it matches the class with higher data


# Hstack takes two arrays (1D or 2D or nD  But both arrays must be of same dimensions) and stacks them horizontally
  data = np.hstack((x,np.reshape(y, (len(y),1))))

  return data, x, y


In [ ]:
print(len(train[train["class"]== 1])) #gamma
print(len(train[train["class"]== 0])) #Hydron

In [ ]:
# We want to stabilize the number of hydron train data and the gamma train data so that the training is not imbalanced
# This is called as oversampling of data
train, x_train, y_train = scale_dataset(train, oversample=True)
valid, x_valid, y_valid = scale_dataset(valid)
test, x_test, y_test = scale_dataset(test)

# kNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report


In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(x_train,y_train)


In [ ]:
y_pred = knn_model.predict(x_test)


In [ ]:
y_pred


In [ ]:
y_test

In [ ]:
print(classification_report(y_test,y_pred))

# **Naive Bayes**

In [ ]:
from sklearn.naive_bayes import GaussianNB


In [ ]:
nb_model = GaussianNB()
nb_model = nb_model.fit(x_train,y_train)


In [ ]:
y_pred = nb_model.predict(x_test)
print(classification_report(y_test,y_pred))

# Logistic **Regression**

In [ ]:
from sklearn.linear_model import LogisticRegression


In [ ]:
lg_model = LogisticRegression()
lg_model = lg_model.fit(x_train,y_train)

In [ ]:
y_pred = lg_model.predict(x_test)
print(classification_report(y_test,y_pred))

# **SVM**

If the SVM accuracy is bad this means that the data has outliers. Making it impossible to assume a line that can clearly classify the binary classification.


In [ ]:
from sklearn.svm import SVC

In [ ]:
svm_model = SVC()
svm_model = svm_model.fit(x_train,y_train)


In [ ]:
y_pred = svm_model.predict(x_test)
print(classification_report(y_test,y_pred))

# **Neural Networks**

In [ ]:
import tensorflow as tf

In [ ]:
def plot_history(history):
  fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
  ax1.plot(history.history['loss'], label='loss')
  ax1.plot(history.history['val_loss'], label='val_loss')
  ax1.set_xlabel('Epoch')
  ax1.set_ylabel('Binary crossentropy')
  ax1.grid(True)

  ax2.plot(history.history['accuracy'], label='accuracy')
  ax2.plot(history.history['val_accuracy'], label='val_accuracy')
  ax2.set_xlabel('Epoch')
  ax2.set_ylabel('Accuracy')
  ax2.grid(True)

  plt.show()

In [ ]:
def train_model(x_train,y_train,num_nodes,dropout_prob,lr,batch_size,epochs):
  nn_model = tf.keras.Sequential([
      tf.keras.layers.Dense(num_nodes, activation='relu', input_shape=(10,)), #Input shape is the shape of the dataset (It is consists of 10 columns)
      tf.keras.layers.Dropout(dropout_prob),
      tf.keras.layers.Dense(num_nodes, activation='relu'),
      tf.keras.layers.Dropout(dropout_prob),
      tf.keras.layers.Dense(1,activation='sigmoid')
  ])
  nn_model.compile(optimizer=tf.keras.optimizers.Adam(lr),loss = 'binary_crossentropy',
                   metrics=['accuracy'])
  history = nn_model.fit(x_train,y_train,epochs = epochs, batch_size = batch_size,validation_split=0.2, verbose=0)

  return nn_model,history

  #NN splits the train set itself into 2 parts training set and validation set thats why we add validation split ratio
  #Verbose is set to 0 so that it doesnt print each and every epochs
  # “Neural networks store the training history of each epoch, which helps us evaluate model performance and tune hyperparameters. While this history doesn’t directly tell us the best number of neurons, it guides us in adjusting the architecture and training setup to achieve better accuracy.”

In [ ]:
#Epoch: they represent how many times the model has seen the entire dataset

least_val_loss = float('inf') #Initial loss is infinity
least_loss_model = None
epochs = 100
for num_nodes in [16,32,64]:
  for dropout_prob in [0,0.2]: # Dropout probability is the prob that a random neuron will be left out during training to prevent overfitting
    for lr in [0.01,0.005,0.001]:
      for batch_size in [32,64,128]:
        print((f"{num_nodes} nodes, dropout {dropout_prob}, lr {lr}, batch size {batch_size}"))
        model, history = train_model(x_train,y_train,num_nodes,dropout_prob,lr,batch_size,epochs)
        plot_history(history)
        val_loss = model.evaluate(x_valid,y_valid)[0]
        #runs the model on the validation set and returns the loss value. This helps track how well the model generalizes to unseen data by monitoring validation loss.
        if val_loss < least_val_loss:
          least_val_loss = val_loss
          least_loss_model = model


In [ ]:
y_pred = least_loss_model.predict(x_test)
y_pred = (y_pred > 0.5).astype(int).reshape(len(y_pred),)
#This line converts probability predictions into binary class labels (0 or 1) using a 0.5 threshold and flattens the result into a 1D array.

In [ ]:
print(classification_report(y_test, y_pred))